Dorothee MG

Study: Validation of empathy videos\
Perceivers watch videos of targets talking about emotional life events. \
Perceivers rate the video on a continuous scale through time. \
We have ECG data from targets (while they talk) and from perceivers (while they watch) 
    
Videos are about 2min long\
We have 5 videos for each emotion

Goal: \
#1 Clean ECG data (e.g., remove artefactual areas, ectopic heart beats, etc.)\
#2 Graph data
    

In [1]:

# libraries

import json
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore


Prepare lists to use later

In [2]:

# targets
targets = ['P03_joy', 'P29_joy', 'P34_joy', 'P42_joy', 'P44_joy',
           'P12_sadness', 'P15_sadness', 'P27_sadness', 'P34_sadness', 'P44_sadness', 
           'P03_anger', 'P15_anger', 'P22_anger', 'P38_anger', 'P42_anger',
           'P17_pride', 'P22_pride', 'P29_pride', 'P42_pride', 'P44_pride',
           'P03_shame', 'P22_shame', 'P37_shame', 'P38_shame', 'P45_shame', 
           'P12_neutral', 'P17_neutral', 'P29_neutral', 'P34_neutral', 'P38_neutral']

# perceivers
perceivers = ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S11", "S15", "S18", "S20", "S23",
              "S24", "S28", "S34", "S37", "S42", "S44", "S45", "S47", "S49", "S51", "S54", "S55", "S57", "S64", "S65"]

# data folders
data_folder = r"physiological_data"


Functions to clean the data -> exclude 'invalid areas' (artefactual) and calculate R-R intervals

Also smooth the data with a 10s rolling window (method: mean)

In [3]:
# helper functions

# determines if two intervals overlap (from biotop) => this will help to remove invalid areas (e.g., artefacts) from data (later)
def does_overlap(intv1, intv2):
    (a1, b1) = intv1
    if a1 > b1: (a1, b1) = (b1, a1)
    (a2, b2) = intv2
    if a2 > b2: (a2, b2) = (b2, a2)
    overlapmin = max([a1, a2])
    overlapmax = min([b1, b2])
    return overlapmin <= overlapmax # true if overlap

# filter out any interval overlapping with invalid area
def exclude_invalid(peak_times, invalid_areas):
    valid_intervals = []
    for i in range(len(peak_times) - 1):  # -1 so it does not go out of range
        start_time = peak_times[i]  # peak 1
        end_time = peak_times[i + 1]  # peak 2
        interval = (start_time, end_time)
        if any(does_overlap(interval, invalid_area) for invalid_area in invalid_areas):
            valid_intervals.append((start_time, np.nan)) # nan to break the line *******
        else:
            valid_intervals.append(interval)  # if not overlap with invalid area, add to list
    return valid_intervals

# define peak times and RR intervals
def calculate_valid_RRs(ecg_data, label):
    peak_times = np.array([data['t'] for data in ecg_data['peaks'][label]]) # identify peak times
    peak_times.sort() # sort in order
    invalid_areas = [(data['start'], data['end']) for data in ecg_data['invalid'][label]] # identify invalid areas
    valid_intervals = exclude_invalid(peak_times, invalid_areas) # exclude invalid
    time_diffs = [end - start if ecg_data['peaks'][label][i]['valid'] and
                  ecg_data['peaks'][label][i-1]['valid'] else None
                  for i, (start, end) in enumerate(valid_intervals)] # identify RR intervals
    peak_times = peak_times[:min(len(peak_times[:-1]), len(time_diffs))]   # peaks - 1 so n peaks = n intervals (otherwise later code does not work properly; maybe there is a better solution?)
    time_diffs = np.array([np.nan if x is None else x for x in time_diffs]) # convert 'none' to 'nan' (useful for correlations later)
    return peak_times, time_diffs # return list of peaks (peak_times) and RR intervals (time_diffs)

# load ECG data from JSON
def load_ecg_data(file_path):
    try:
        with open(file_path, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        return {} # just in case

# clean perceiver ECG data => separate from target cleaning, because different structure (file name, channel name, etc.)
def clean_perceiver_ecg(data_folder, perceiver_id, target):
    perceiver_file = os.path.join(data_folder, f"{perceiver_id}_{target}_peaks.json") # open perceiver file
    if not os.path.exists(perceiver_file):
        return [], [] # (perceivers did not watch all targets) 
    ecg_data_perceiver = load_ecg_data(perceiver_file)
    peak_times_perceiver, time_diffs_perceiver = calculate_valid_RRs(ecg_data_perceiver, "ECG, X, RSPEC-R") # calculate RR intervals
    return peak_times_perceiver, time_diffs_perceiver

# clean target ECG data => ** I should combine this with clean_perceiver_ecg at some point
def clean_target_ecg(data_folder, target):
    target_file = os.path.join(data_folder, f"{target}_peaks.json") # open target file
    ecg_data_target = load_ecg_data(target_file)
    target_id = target.split('_')[0] # take target number (before the '_'), not whole video name
    peak_times_target, time_diffs_target = calculate_valid_RRs(ecg_data_target, f"ECG {target_id}") # calculate RR intervals
    return peak_times_target, time_diffs_target

# sliding window of 10s for HRV => but there is less data at the extremities so I'm not sure this works 100%
def sliding_window_mean(peak_times, values):

    averaged_peak_times = []
    averaged_time_diffs = []
    half_window = 5  # half of the 10s window (5s before, 5s after)

    for peak in peak_times:
        start_time, end_time = peak - half_window, peak + half_window # define the window
        window_values = values[(peak_times >= start_time) & (peak_times <= end_time)] # extract values that fall within the window
        if len(window_values) > 0:  # avoid empty (just in case)
            averaged_peak_times.append(peak)
            averaged_time_diffs.append(np.nanmean(window_values))  # compute mean ignoring nan

    return averaged_peak_times, averaged_time_diffs


Visualize data, and at the same time explore different strategies of ectopic beat detection. 


In [ ]:


count = 0


for target in targets:
    for perceiver in perceivers:
        
        
        if count >= 20:
            break # I added this to not produce all plots (28 perceivers * 12 targets) otherwise the file becomes too big
        
        
        ecg_data_perceiver = load_ecg_data(os.path.join(data_folder, f"{perceiver}_{target}_peaks.json")) # load data for later (already loaded in clean_perceiver_ecg could be optimized)
        perceiver_peak_times, perceiver_time_diffs = clean_perceiver_ecg(data_folder, perceiver, target) # extract peak times and RR intervals
        
        if len(perceiver_time_diffs) == 0:
            continue # skip if no data (just in case)
        
        print(f"RR-intervals: {perceiver_time_diffs}") # print the RR intervals (could be useful to check anomalies)
        
        invalid_areas = ecg_data_perceiver.get("invalid", {}).get("ECG, X, RSPEC-R", []) # identify invalid areas

        smoothed_RR_times, smoothed_RR_values = sliding_window_mean( # smoothed RR intervals
            np.array(perceiver_peak_times), np.array(perceiver_time_diffs))
        
        
        
        # malik rule: RR-intervals differing by more than the 20 % of the next RR interval are removed.
        outlier_malik = [RR for RR in range(len(perceiver_time_diffs) - 1) if abs(perceiver_time_diffs[RR] - perceiver_time_diffs[RR + 1]) > 0.2 * perceiver_time_diffs[RR]] # -1 otherwise out of bounds?
        
        # kamath rule: R-R intervals increasing or decreasing by more than 32.5% or by more than 24.5% from the previous interval are removed.
        outlier_kamath = [RR for RR in range(len(perceiver_time_diffs) - 1) if 0 < perceiver_time_diffs[RR + 1] - perceiver_time_diffs[RR] > .325 * perceiver_time_diffs[RR] or 
                                                                                0 < (perceiver_time_diffs[RR] - perceiver_time_diffs[RR+1] > .245 * perceiver_time_diffs[RR])]
                 
        # karlsson rule: RR-intervals differing by more than the 20 % of the mean of previous and next RR-interval are removed.
        outlier_karlsson = [RR for RR in range(1, len(perceiver_time_diffs) - 1) if abs(((perceiver_time_diffs[RR - 1] + perceiver_time_diffs[RR + 1]) / 2) - perceiver_time_diffs[RR]) > 0.2 * ((perceiver_time_diffs[RR - 1] + perceiver_time_diffs[RR + 1]) / 2)]
        
        # acar rule: RR-intervals differing by more than the 20 % of the mean of last 9 RR intervals are removed.
        outlier_acar = [RR for RR in range(9, len(perceiver_time_diffs)) if abs(perceiver_time_diffs[RR] - np.mean(perceiver_time_diffs[RR-9:RR])) > 0.2 * np.mean(perceiver_time_diffs[RR-9:RR])]

        # sd rule: RR-intervals exceeding 3 SD from the mean of all RR-intervals are removed. 
        outlier_sd = [i for i in range(len(perceiver_time_diffs) -1) if abs(zscore(perceiver_time_diffs)[i]) > 3]
    
    
    
        plt.figure(figsize=(12, 4)) # create plot

        plt.plot(perceiver_peak_times, perceiver_time_diffs, color='black', linestyle='-', marker='o', markersize=3) # raw RR intervals
        plt.plot(smoothed_RR_times, smoothed_RR_values, color='red', linewidth=1, linestyle='--') # smoothed RR intervals
        for invalid in invalid_areas: # highlight invalid areas in ECG
            plt.axvspan(invalid["start"], invalid["end"], color="grey", alpha=0.3)
            
        for i in outlier_malik: # identify ectopic beats according to each rule (different color and shape)
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='royalblue', s=200, marker='o')            
        for i in outlier_kamath:
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='green', s=150, marker='^')
        for i in outlier_karlsson:
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='red', s=100, marker='s')
        for i in outlier_acar:
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='magenta', s=70, marker='D')
        for i in outlier_sd:
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='orange', s=50, marker='X')
            
            
            
        plt.xlabel('time (s)')
        plt.ylabel('RR intervals')
        plt.title(f'RR intervals: {perceiver} evaluating {target}')

        plt.tight_layout()
        plt.show()
        
        
        
        count += 1
    if count >= 20:
        break        